# Práctica: Introducción a modelos de visión con *timm* y comparación de arquitecturas

## 1. Introducción

En esta práctica se introduce el uso de modelos de visión por computador preentrenados mediante la librería timm.

Se trabajará con técnicas de *transfer learning* y se compararán distintas arquitecturas en tareas de clasificación de imágenes.

La práctica se divide en:

* **Parte 1 (tutorial guiado)** → dataset sencillo
* **Parte 2 (práctica)** → problema más realista

## 2. Objetivos de aprendizaje

* Utilizar modelos preentrenados
* Adaptar modelos a nuevas tareas
* Comprender *feature extraction* vs *fine-tuning*
* Comparar arquitecturas CNN y Transformers
* Analizar el impacto del preprocesado

# 3. Parte 1: Tutorial (CIFAR-10)

## 3.1 Importaciones

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import timm

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


## 3.2 Dataset

Se utilizará CIFAR-10.

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_dataset = datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)

test_dataset = datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

100%|██████████| 170M/170M [00:04<00:00, 36.2MB/s]


## 3.3 Modelo

In [ ]:
model = timm.create_model(
    "resnet18",
    pretrained=True,
    num_classes=10
).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

# 3.4 Entrenamiento

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

def train_one_epoch():
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

def evaluate():
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

## 3.5 Ejecutar

In [ ]:
for epoch in range(3):
    loss = train_one_epoch()
    acc = evaluate()
    print(f"Epoch {epoch+1} | Loss {loss:.4f} | Acc {acc:.4f}")


Epoch 1 | Loss 0.4048 | Acc 0.9105
Epoch 2 | Loss 0.1636 | Acc 0.9283
Epoch 3 | Loss 0.0945 | Acc 0.9362


## 3.6 Reflexión

* ¿Qué accuracy se obtiene?
* ¿Por qué converge rápido?
* ¿Qué aporta `pretrained=True`?

# 4. Parte 2: Práctica (Gatos vs Perros)

## 4.1 Dataset

Se utilizará el dataset Oxford-IIIT Pet.

Se transformará en un problema binario:

* 0 → gato
* 1 → perro

## 4.2 Descarga y preparación

In [ ]:
from torchvision import datasets
from torch.utils.data import DataLoader

# Transformaciones base
train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_full = datasets.OxfordIIITPet(
    root="./data",
    split="trainval",
    target_types="category",
    download=True,
    transform=train_tfms
)

val_full = datasets.OxfordIIITPet(
    root="./data",
    split="test",
    target_types="category",
    download=True,
    transform=val_tfms
)

# Conversión a binario
def to_binary(y):
    return 0 if y < 12 else 1

def collate_fn(batch):
    xs, ys = zip(*batch)
    xs = torch.stack(xs)
    ys = torch.tensor([to_binary(int(y)) for y in ys])
    return xs, ys

train_loader = DataLoader(train_full, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_full, batch_size=32, collate_fn=collate_fn)

100%|██████████| 792M/792M [00:29<00:00, 26.7MB/s]
100%|██████████| 19.2M/19.2M [00:01<00:00, 12.5MB/s]


## 4.3 Modelos a comparar

* resnet18
* vit_tiny_patch16_224
* swin_tiny_patch4_window7_224

## 4.4 Funciones base

In [ ]:
def create_model(name):
    return timm.create_model(name, pretrained=True, num_classes=2).to(device)

def set_trainable(model, freeze):
    if freeze:
        for p in model.parameters():
            p.requires_grad = False

        if hasattr(model, "head"):
            for p in model.head.parameters():
                p.requires_grad = True
        elif hasattr(model, "fc"):
            for p in model.fc.parameters():
                p.requires_grad = True
    else:
        for p in model.parameters():
            p.requires_grad = True


def get_optimizer(model):
    params = [p for p in model.parameters() if p.requires_grad]
    return optim.AdamW(params, lr=3e-4)

## 4.5 Entrenamiento

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

## 4.6 Experimento

In [ ]:
model_name = "resnet18"  # cambiar

model = create_model(model_name)

freeze = True  # probar True / False
set_trainable(model, freeze)

optimizer = get_optimizer(model)
criterion = nn.CrossEntropyLoss()

for epoch in range(3):
    loss = train_one_epoch(model, train_loader, optimizer, criterion)
    acc = evaluate(model, val_loader)
    print(f"{model_name} | freeze={freeze} | Acc {acc:.4f}")


resnet18 | freeze=True | Acc 0.6814
resnet18 | freeze=True | Acc 0.7438
resnet18 | freeze=True | Acc 0.7441


## 4.7 Tareas

Para cada modelo:

1. Ejecutar con `freeze=True`
2. Ejecutar con `freeze=False`
3. Comparar resultados

## 4.8 Análisis

Responder:

* ¿Qué modelo funciona mejor?
* ¿Cuál converge más rápido?
* ¿Qué impacto tiene el fine-tuning?
* ¿Diferencias entre CNN y Transformers?


# 5. Parte opcional

## Objetivo

Mejorar el rendimiento modificando el preprocesado.

## Ejemplo

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

## Requisito

* Implementar una mejora
* Comparar con baseline
* Justificar resultados
* Tabla de resultados
* Análisis breve